In [ ]:
import shutil
import random
import yaml
from pathlib import Path
import sys
sys.path.append(str(Path("..").resolve()))
data_dir = Path("../data")
import json
from collections import defaultdict

In [ ]:
SPECIES_MAP = {
    "Sus scrofa (Wild boar)":         0,
    "Cervus elaphus (Red deer)":      1,
    "Capreolus capreolus (Roe deer)": 2,
}


In [ ]:
def compute_flight_species(ann_dir: Path) -> dict:
    flight_species = {}
    for ann_file in sorted(ann_dir.glob("*_gt.txt")):
        flight_id = ann_file.stem.replace("_gt", "")
        present = set()
        with open(ann_file, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 13:
                    continue
                if int(parts[12]) == 1 or float(parts[8]) < 0.3:
                    continue
                sp = parts[9].strip()
                if sp in SPECIES_MAP:
                    present.add(sp)
        flight_species[flight_id] = present
    return flight_species


In [ ]:
def stratified_flight_split(flight_species: dict, ratio=(0.70, 0.15, 0.15), seed=42) -> dict:
    random.seed(seed)
    groups = defaultdict(list)
    for fid, species in flight_species.items():
        sig = frozenset(s.split("(")[1].rstrip(")") for s in species)
        groups[sig].append(fid)
 
    train, val, test = [], [], []
    for sig, fids in sorted(groups.items()):
        fids = sorted(fids)
        random.shuffle(fids)
        n = len(fids)
        t_end = max(1, int(n * ratio[0]))
        v_end = t_end + max(0, int(n * ratio[1]))
        train += fids[:t_end]
        val   += fids[t_end:v_end]
        test  += fids[v_end:]
 
    return {"train": train, "val": val, "test": test}

In [ ]:
def build_yolo_structure(data_dir, output_dir, splits):
    for split, flight_ids in splits.items():
        (output_dir / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_dir / split / 'labels').mkdir(parents=True, exist_ok=True)
        for flight_id in flight_ids:
            flight_labels_dir = data_dir / "labels" / flight_id
            flight_images_dir = data_dir / "frames_detection" / flight_id / "thermal"
            if not flight_labels_dir.exists() or not flight_images_dir.exists():
                print(f"Missing dirs for flight {flight_id}, skipping")
                continue
            for label_file in flight_labels_dir.glob("*.txt"):
                image_file = flight_images_dir / f"{label_file.stem}.png"
                if image_file.exists():
                    shutil.copy(label_file, output_dir / split / "labels" / label_file.name)
                    shutil.copy(image_file, output_dir / split / "images" / image_file.name)
 
    yaml_content = {
        "path": str(output_dir.resolve()),
        "train": "train/images",   
        "val": "val/images",
        "test": "test/images",
        "nc": 3,                  
        "names": {
            0: "Wild boar",
            1: "Red deer",
            2: "Roe deer"
        }
    }
    yaml_path = output_dir / "dataset.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_content, f, sort_keys=False)
 
 
flight_species = compute_flight_species(data_dir)
print(f"Flights found: {len(flight_species)}")
 
splits = stratified_flight_split(flight_species, ratio=(0.70, 0.15, 0.15), seed=42)
print(f"Split sizes — train: {len(splits['train'])}, "
      f"val: {len(splits['val'])}, test: {len(splits['test'])}")
 
build_yolo_structure(data_dir, data_dir / 'yolo_data', splits)


In [ ]:
# 1. Count Total Labels for Flight 0
labels_0 = list((data_dir / "labels" / "0").glob("*.txt"))
print(f"Total Label files for Flight 0: {len(labels_0)}")
 
images_0 = list((data_dir / "frames_detection" / "0" / "thermal").glob("*.png"))
print(f"Total Image files for Flight 0: {len(images_0)}")
 
# 3. Check for Overlap 
matches = 0
for label in labels_0:
    expected_image = data_dir / "frames_detection" / "0" / "thermal" / f"{label.stem}.png"
    if expected_image.exists():
        matches += 1
 
print(f"\nTotal Overlapping Matches: {matches}")

In [ ]:
def assign_video_and_frame_ids(coco_data, splits):
    all_flight_ids = [fid for ids in splits.values() for fid in ids]
    flight_to_video_id = {fid: i + 1 for i, fid in enumerate(sorted(all_flight_ids, key=str))}
 
    images_by_flight = defaultdict(list)
    for img in coco_data["images"]:
        stem = Path(img["file_name"]).stem            # "{flight_id}_{frame_idx:08d}"
        flight_id = "_".join(stem.split("_")[:-1])
        frame_idx = int(stem.split("_")[-1])           # original (sparse) frame number
        images_by_flight[flight_id].append((frame_idx, img))
 
    for flight_id, entries in images_by_flight.items():
        entries.sort(key=lambda x: x[0])                # chronological order
        video_id = flight_to_video_id.get(flight_id)
        if video_id is None:
            print(f"  ⚠ flight {flight_id} not found in any split — skipping")
            continue
        for frame_id, (frame_idx, img) in enumerate(entries):
            img["video_id"]       = video_id
            img["frame_id"]       = frame_id      # 0-indexed sequence position
            img["orig_frame_idx"] = frame_idx     # kept for traceability
 
    return flight_to_video_id


In [ ]:
def build_coco_splits(json_path, splits, output_path):
    with open(json_path) as f:
        coco_data = json.load(f)
 
    flight_to_video_id = assign_video_and_frame_ids(coco_data, splits)
 
    img_by_stem = {
        Path(img["file_name"]).stem: img
        for img in coco_data["images"]
    }
 
    ann_by_img = defaultdict(list)
    for ann in coco_data["annotations"]:
        ann_by_img[ann["image_id"]].append(ann)
 
    output_path.mkdir(parents=True, exist_ok=True)
 
    for split, flight_ids in splits.items():
        flight_set = set(flight_ids)
        split_imgs, split_anns = [], []
 
        for stem, img in img_by_stem.items():
            flight_id = "_".join(stem.split("_")[:-1])
            if flight_id not in flight_set:
                continue
            split_imgs.append(img)
            split_anns.extend(ann_by_img[img["id"]])
 
        # sort by (video_id, frame_id) 
        split_imgs.sort(key=lambda im: (im["video_id"], im["frame_id"]))
 
        split_coco = {
            "images": split_imgs,
            "annotations": split_anns,
            "categories": coco_data["categories"],
            "videos": [
                {"id": vid, "file_name": fid}
                for fid, vid in sorted(flight_to_video_id.items(), key=lambda x: x[1])
                if fid in flight_set
            ],
        }
        out_file = output_path / f"instances_{split}.json"
        with open(out_file, "w") as f:
            json.dump(split_coco, f)
        print(f"{split}: {len(split_imgs)} images, {len(split_anns)} annotations, "
              f"{len(split_coco['videos'])} videos")
 
 
build_coco_splits(
    data_dir / "annotations" / "instances_default.json",
    splits,
    data_dir / "annotations"
)